In [ ]:
!pip install -q sentencepiece==0.2.0 bert-score==0.3.13 nltk==3.9.1 sacremoses 2>/dev/null

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 32.4 MB/s eta 0:00:00


In [ ]:
import importlib, sys
try:
    import torch
except ImportError:
    !pip install -q torch

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("Dependencies ready.")

Dependencies ready.


In [ ]:
import os, math, time, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu


In [ ]:
TRAIN_SA = "/content/train_sa_10000.csv"
TRAIN_EN = "/content/train_en_10000.csv"
DEV_SA   = "/content/dev_sa_1000.csv"
DEV_EN   = "/content/dev_en_1000.csv"
TEST_SA  = "/content/test_sa_1000.csv"
TEST_EN  = "/content/test_en_1000.csv"

# ---------------- HYPERPARAMETERS ----------------
CFG = dict(
    src_vocab   = 8000,
    tgt_vocab   = 8000,
    d_model     = 256,
    n_heads     = 8,
    n_enc       = 4,
    n_dec       = 4,
    d_ff        = 1024,
    dropout     = 0.3,
    max_len     = 128,
    batch_size  = 64,
    lr          = 1.0,
    warmup      = 4000,
    label_smooth= 0.1,
    epochs      = 40,
    patience    = 6,
    grad_clip   = 1.0,
    beam_size   = 5,
    length_alpha= 0.7,
)


PAD, BOS, EOS, UNK = 0, 1, 2, 3
for k, v in CFG.items():
    print(f"{k:12s}: {v}")

src_vocab   : 8000
tgt_vocab   : 8000
d_model     : 256
n_heads     : 8
n_enc       : 4
n_dec       : 4
d_ff        : 1024
dropout     : 0.3
max_len     : 128
batch_size  : 64
lr          : 1.0
warmup      : 4000
label_smooth: 0.1
epochs      : 40
patience    : 6
grad_clip   : 1.0
beam_size   : 5
length_alpha: 0.7


### Download Data Files

To resolve the `FileNotFoundError`, we need to download the required CSV files. Assuming they are available from a public source, you can use `wget` to download them. Please replace the placeholder URLs below with the actual links if they are different.

In [ ]:
# The data files are already present in /content/, so no download is necessary.
# This cell is now empty as the files are already in the correct path.
# If you need to download them from a specific source, please update the paths in cell NLLWpVG-BX2R and uncomment/update the wget commands with the correct URLs.

In [ ]:
def load_pairs(sa_path, en_path, drop_empty=True):
    sa = pd.read_csv(sa_path)
    en = pd.read_csv(en_path)
    df = sa.merge(en, on="Source_id", how="inner")
    df["Sentence_sa"] = df["Sentence_sa"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df["Sentence_en"] = df["Sentence_en"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    if drop_empty:
        df = df[(df["Sentence_sa"].str.len() > 0) & (df["Sentence_en"].str.len() > 0)]
    return df.reset_index(drop=True)

train_df = load_pairs(TRAIN_SA, TRAIN_EN)
dev_df   = load_pairs(DEV_SA, DEV_EN)


print("Train pairs:", len(train_df))
print("Dev   pairs:", len(dev_df))
train_df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'train_sa_10000.csv'